In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, "..", "..", ".."))  # activate docs/

In [ ]:
using LinearAlgebra, SparseArrays, Qritical

L   = 8
dof = SpinHalf()
g   = Chain(L)
ops = algebra_generators(dof)

# A generic test state: random statevector → left-canonical MPS (D ≤ 16 for L=8)
ψ_rand = as_state(randn(ComplexF64, 2^L), fill(2, L))
mps    = to_mps(ψ_rand; trunc=MaxBondDimTrunc(16), form=:left)
println("MPS  L=$L  D=", maximum(size(t.data, 3) for t in mps.tensors))

# Ex 6. Matrix Product Operators (MPOs)

## (a) XXZ MPO via `MPO(XXZ(Chain(L)))`

The `MPO` constructor encodes the Hamiltonian as a finite-state machine (FSM)
over the auxiliary bond.  Bond dimension is 5 for the XXZ chain:
`IdL → S⁺ → S⁻ → Sᶻ → IdR`.

In [ ]:
H_xxz = XXZ(g; J=1.0, Jz=1.0, h=0.0)
W_xxz = MPO(H_xxz)

# MPO tensors are raw order-4 arrays (χ_L, d_out, d_in, χ_R)
println("MPO bond dimensions: ", [size(t, 1) for t in W_xxz.tensors], " → ",
        size(W_xxz.tensors[end], 4))
println("Interior W tensor shape: ", size(W_xxz.tensors[L÷2]))

In [ ]:
# Physical check: L=2 Heisenberg dimer has eigenvalues {−¾, ¼, ¼, ¼}
H2   = XXZ(Chain(2); J=1.0, Jz=1.0, h=0.0)
M2   = matrix_repr(H2)         # 4×4 dense matrix via direct construction
evs2 = sort(real.(eigvals(M2)))
println("L=2 eigenvalues: ", round.(evs2; sigdigits=4))
println("Expected:        [-0.75, 0.25, 0.25, 0.25]")

In [ ]:
# Hermiticity and sparse structure of the full L=6 matrix
H6   = XXZ(Chain(6); J=1.0, Jz=1.0, h=0.0)
M_sp = matrix_repr(H6, SparseFormat())
M_dn = matrix_repr(H6, DenseFormat())

println("Dense: ", size(M_dn), "  Sparse nnz: ", nnz(M_sp), " / ", prod(size(M_sp)))
println("Hermitian: ", norm(M_dn - M_dn') < 1e-13)

## (b) All-to-all $\hat S^z_{\mathrm{tot}}^2 = \left(\sum_i S^z_i\right)^2$

The total-$S^z$ squared is itself a `LatticeOperator` built via
`total_magnetization` (returns $\hat S^z_{\mathrm{tot}}$).  Its square is an
all-to-all $\sum_{i,j} S^z_i S^z_j$ operator.

In [ ]:
# Build the staggered and total magnetization via Qritical operators
M_stag = staggered_magnetization(g; dof=dof)   # Σᵢ (−1)ⁱ Sᶻᵢ
M_tot  = total_magnetization(g; dof=dof)         # Σᵢ Sᶻᵢ

W_stag = MPO(M_stag)
W_tot  = MPO(M_tot)

println("Staggered mag MPO bond dim: ", [size(t, 1) for t in W_stag.tensors])
println("Total mag MPO bond dim: ",     [size(t, 1) for t in W_tot.tensors])

In [ ]:
# For the all-up state |↑↑…↑⟩, Stot = L/2 so ⟨Stot⟩ = L/2.
# |↑⟩ is basis state 1, so the all-up statevector is e₁ (kron index 1).
v_up     = zeros(ComplexF64, 2^L);  v_up[1] = 1.0
all_up_c = to_mps(as_state(v_up, fill(2, L)); form=:left)

Stot_up = real(expect(all_up_c, W_tot))
println("⟨↑↑…↑|Stot|↑↑…↑⟩ = ", round(Stot_up; sigdigits=6), "  (expected ", L/2, ")")

## (c) Expectation value $\langle \psi | H | \psi \rangle$ via MPO

`expect(mps, mpo)` sweeps a three-legged environment (bra × MPO × ket) from
left to right — cost $O(L\chi^2 d^2 \chi_W)$.

In [ ]:
E_xxz = expect(mps, W_xxz)
println("⟨ψ|H_XXZ|ψ⟩ = ", round(real(E_xxz); sigdigits=8))

# ⟨H⟩ is gauge-invariant: recompute from a right-canonical copy of the same state
E_xxz2 = expect(canonicalize(mps, RightCanonical()), W_xxz)
println("Same ⟨H⟩ from right-canonical: ", round(real(E_xxz2); sigdigits=8))

In [ ]:
# Identity MPO: ⟨ψ|I|ψ⟩ = ⟨ψ|ψ⟩
I_op  = identity_operator(g, dof)
W_Id  = MPO(I_op)
E_Id  = expect(mps, W_Id)
norm2 = real(overlap(mps, mps))
println("⟨ψ|I|ψ⟩ = ", round(E_Id;  sigdigits=8))
println("⟨ψ|ψ⟩   = ", round(norm2; sigdigits=8))
@assert abs(E_Id - norm2) < 1e-12 "Identity MPO should equal norm squared"
println("Identity MPO check ✓")

## (d) Applying MPO to MPS: $|\phi\rangle = H|\psi\rangle$

`apply_mpo(mpo, mps)` contracts each W tensor with the corresponding site tensor,
producing a new MPS with expanded bond dimension $\chi_{\mathrm{new}} = \chi \cdot \chi_W$.
The result can be truncated immediately via the `trunc` keyword.

In [ ]:
Hpsi     = apply_mpo(W_xxz, mps)
Hpsi_can = canonicalize(Hpsi, LeftCanonical())

println("‖H|ψ⟩‖² = ", round(real(overlap(Hpsi_can, Hpsi_can)); sigdigits=8))
println("Bond dims of H|ψ⟩: ", [size(t.data,3) for t in Hpsi_can.tensors])

In [ ]:
# Verify: ‖H|ψ⟩‖² = ⟨ψ|H²|ψ⟩  (if H is Hermitian)
norm2_Hpsi = real(overlap(Hpsi_can, Hpsi_can))
HHpsi      = apply_mpo(W_xxz, Hpsi_can)
HHpsi_c    = canonicalize(HHpsi, LeftCanonical())
E_H2_via_apply = real(overlap(mps, HHpsi_c))  # ⟨ψ|H²|ψ⟩
println("‖H|ψ⟩‖² via norm:      ", round(norm2_Hpsi;      sigdigits=8))
println("⟨ψ|H²|ψ⟩ via double-apply: ", round(E_H2_via_apply; sigdigits=8))

In [ ]:
# Truncated application
Hpsi_trunc = apply_mpo(W_xxz, mps; trunc=MaxBondDimTrunc(8))
Hpsi_tr_c  = canonicalize(Hpsi_trunc, LeftCanonical())
println("Truncated bond dims (D=8): ", [size(t.data,3) for t in Hpsi_tr_c.tensors])
trunc_fid  = abs(overlap(Hpsi_tr_c, Hpsi_can))^2 /
             (real(overlap(Hpsi_tr_c, Hpsi_tr_c)) * real(overlap(Hpsi_can, Hpsi_can)))
println("Truncation fidelity: ", round(trunc_fid; sigdigits=4))